Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## First call to a model

- Every call is the same three steps : build the model, invoke it, read the reply
- Only the import and the class change if we swap provider

Sends three different queries and times each one.

In [ ]:
# import libraries
from langchain_openai import ChatOpenAI


# create the client, pointed at local Ollama
llm = make_llm()


def run_query(text: str) -> str:
    print(f"Query: {text}")
    response = llm.invoke(text)  # <- Replace the prompt here
    return response.content


# simple prompt and request to the LLM API
response = run_query("Write a short shift handover note for the night team.")  # <- Replace the prompt here
print("Model response:\n")
print(response)


print()

### Solution Part 1 : execution-time measurement across 3 queries
This gives us time, character count, and word count per query, so we can compare length and structure directly, which is what the exercise asks us to observe. (Reminder: run one query first as a warm-up, or the first timing includes the model-load time and looks artificially slow.)

In [ ]:
import time

queries = [
    "Write a short shift handover note for the night team.",
    "Explain in 3 sentences why a depot would run two shifts.",
    "List 5 popular Python libraries for data science.",
]

for i, q in enumerate(queries, start=1):
    start = time.time()
    response = llm.invoke(q)
    elapsed = time.time() - start

    text = response.content
    print(f"--- Query {i}: {q}")
    print(f"Time: {elapsed:.2f} s | Characters: {len(text)} | Words: {len(text.split())}")
    print(text)
    print()

### Solution Part 2 : the run_query helper
And the two parts combined : the helper timed across all three queries, which is the clean version worth putting in the notebook:

In [ ]:
def run_query(text: str) -> str:
    """Send a prompt to the LLM and return just the response text."""
    response = llm.invoke(text)
    return response.content

print(run_query("Write a short shift handover note for the night team."))

import time

def run_query(text: str) -> str:
    """Send a prompt to the LLM and return just the response text."""
    return llm.invoke(text).content

queries = [
    "Write a short shift handover note for the night team.",
    "Explain in 3 sentences why a depot would run two shifts.",
    "List 5 popular Python libraries for data science.",
]

for i, q in enumerate(queries, start=1):
    start = time.time()
    answer = run_query(q)
    elapsed = time.time() - start
    print(f"--- Query {i} ({elapsed:.2f} s, {len(answer)} chars) ---")
    print(answer, "\n")


### Try a much longer prompt

- Paste a few paragraphs into one of the queries and time it again
- Compare where the time goes : reading the prompt, or writing the answer